# 🚀 NovaLM-ULTRA Complete Training Notebook
## अपना खुद का LLM बनाएं - Train Your Own Language Model

This notebook covers EVERYTHING:
- 📦 Install dependencies
- 🤖 Create model with ALL parameters
- 📚 Load ANY HuggingFace dataset with filters
- 🔤 Tokenizer (character-level or HuggingFace)
- 🎯 Full training with all features
- ✍️ Text generation
- 💾 Save/Load checkpoints

In [ ]:
# ============================================================================
# SECTION 0: INSTALL DEPENDENCIES (Run once)
# ============================================================================
# Uncomment and run if needed:
# !pip install torch datasets transformers tokenizers tqdm wandb

In [ ]:
# ============================================================================
# SECTION 1: IMPORTS
# ============================================================================
import sys
import os
import math
import time
import json
import random
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple
from dataclasses import dataclass
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# Add NovaLM-ULTRA to path
project_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

# Import NovaLM-ULTRA
try:
    from NovaLM_ULTRA import create_ultra_model, UltraConfig
    HAS_NOVA = True
    print("✅ NovaLM-ULTRA imported!")
except ImportError:
    HAS_NOVA = False
    print("⚠️  NovaLM-ULTRA not found. Using minimal transformer.")

---
## ⚙️ STEP 1: CONFIGURE ALL PARAMETERS

### 📖 Parameter Guide (पैरामीटर गाइड):

| Parameter | Values | Meaning |
|-----------|--------|--------|
| `dim` | 256-4096 | Model size. Higher = more capacity |
| `num_layers` | 4-64 | Number of layers. More = deeper |
| `num_heads` | 4-64 | Attention heads. Must divide dim |
| `n_kv_heads` | 1-64 | GQA KV heads. Lower = less memory |
| `vocab_size` | 1000-128000 | Vocabulary size |
| `block_size` | 128-8192 | Sequence length |
| `weight_share` | 1-16 | Weight sharing iterations |
| `epochs` | 1-100 | Training epochs |
| `batch_size` | 2-64 | Batch size (depends on GPU RAM) |
| `learning_rate` | 1e-5 to 1e-3 | How fast to learn |

**Config Presets:**
- 🏃 **fast**: dim=256, layers=4, heads=8 (testing)
- ⚖️ **balanced**: dim=512, layers=8, heads=16 (small LLM)
- 💪 **powerful**: dim=2048, layers=24, heads=32 (production)
- 🚀 **ultra**: dim=4096, layers=64, heads=64 (SOTA)

In [ ]:
# ============================================================================
# ALL CONFIGURATION PARAMETERS - एक ही जगह सब कुछ
# ============================================================================

# === MODEL NAME ===
MODEL_NAME = "MyNovaLM"  # Your LLM का नाम

# === MODEL ARCHITECTURE ===
DIM = 512                 # Model dimension (256=fast, 512=small, 768=medium, 1024=large)
NUM_LAYERS = 8            # Number of layers (4=shallow, 8=medium, 12=deep, 24=very deep)
NUM_HEADS = 16            # Attention heads (must divide DIM: dim/64 or dim/128)
N_KV_HEADS = None         # GQA KV heads (None=auto, 1=max savings, 4=good, 8=balanced)
FFN_DIM = None            # FFN dim (None=dim*4, higher = more capacity)
VOCAB_SIZE = 10000        # Vocab size (auto-adjusted for tokenizer)
BLOCK_SIZE = 256          # Sequence length (128=fast, 256=default, 512=good, 1024=long)
WEIGHT_SHARE = 4          # Weight sharing (1=off, 4=good, 8=deep, 16=ultra)
WEIGHT_TYING = True       # Share embed/head weights

# === ARCHITECTURE COMPONENTS ===
ATTN_TYPE = "gqa"          # "gqa"=standard, "mla"=deepseek
USE_RWKV = True            # RWKV-7 TimeMix for fast CPU
RWKV_FREQ = 2              # RWKV every N layers
USE_SSM = True             # Selective SSM (Mamba)
SSM_FREQ = 4               # SSM every N layers
SSM_STATE = 16             # SSM state dimension
USE_MEMORY = True          # Titans Neural Memory
MEMORY_SLOTS = 256         # Memory slots (more = more capacity)
MEMORY_TOPK = 16           # Top-k memory retrieval
MEMORY_FREQ = 4            # Memory every N layers
USE_ROUTER = True          # Adaptive Router
ROUTER_DIM = 256           # Router hidden dimension
USE_ROPE = True            # RoPE position embeddings

# === DATASET ===
DATASET_NAME = "roneneldan/TinyStories"  # HF dataset name
DATASET_SUBSET = None                     # Dataset subset (e.g., "sample-10BT")
DATASET_SPLIT = "train"                  # Dataset split
TEXT_COLUMN = "text"                     # Text column name
VAL_SIZE = 0.1                            # Validation fraction (0.1 = 10%)
MAX_SAMPLES = None                        # Max samples (None = all)
MIN_TEXT_LEN = None                       # Filter: minimum text length
MAX_TEXT_LEN = None                       # Filter: maximum text length
STREAMING = False                         # Streaming mode for huge datasets

# === TOKENIZER ===
TOKENIZER_TYPE = "char"   # "char"=simple, "hf"=huggingface
HF_TOKENIZER_NAME = "gpt2"  # HF tokenizer name (if TOKENIZER_TYPE == "hf")

# === TRAINING ===
EPOCHS = 3                # Training epochs (1=test, 3=quick, 10=standard)
BATCH_SIZE = 8            # Batch size (4=small GPU, 8=default, 16=good)
LEARNING_RATE = 3e-4      # Peak learning rate
MIN_LR = 3e-5             # Minimum learning rate
WARMUP_STEPS = 100        # Warmup steps
WEIGHT_DECAY = 0.1        # AdamW weight decay
BETA1 = 0.9               # Adam beta1
BETA2 = 0.95              # Adam beta2
GRAD_CLIP = 1.0           # Gradient clipping norm
DROPOUT = 0.0             # Dropout rate
LABEL_SMOOTHING = 0.0     # Label smoothing
LR_SCHEDULER = "cosine"  # "cosine", "linear", "constant"

# === HARDWARE ===
DEVICE = "auto"           # "auto", "cpu", "cuda", "mps"
NUM_WORKERS = 0           # DataLoader workers (0=Windows safe)
MIXED_PRECISION = "fp16"  # "fp16", "bf16", "no"
USE_COMPILE = False       # torch.compile (20-30% faster)
MULTI_GPU = False         # Use multiple GPUs
GPU_IDS = None            # Specific GPU IDs (e.g., "0,1,2,3")
CPU_THREADS = None        # CPU threads for OMP

# === LOGGING ===
SAVE_DIR = "checkpoints"  # Save directory
SAVE_EVERY = 1000         # Save checkpoint every N steps
LOG_INTERVAL = 10         # Log every N steps
USE_WANDB = False         # Use Weights & Biases
WANDB_PROJECT = "novaultra"  # W&B project

# === GENERATION ===
GENERATE_AFTER_TRAIN = True  # Generate text after training
PROMPT = "Once upon a time"  # Generation prompt
GEN_TOKENS = 100              # Number of tokens to generate
GEN_TEMP = 0.7                # Temperature (0.1=deterministic, 1.0=creative)
GEN_TOP_K = 50                # Top-k sampling
GEN_TOP_P = 0.9               # Top-p nucleus sampling

print("✅ Configuration loaded!")
print(f"   Model: {MODEL_NAME}, dim={DIM}, layers={NUM_LAYERS}, heads={NUM_HEADS}")
print(f"   Dataset: {DATASET_NAME}, Epochs: {EPOCHS}, Batch: {BATCH_SIZE}")
print(f"   Components: GQA={'gqa'}, RWKV={'on' if USE_RWKV else 'off'}, SSM={'on' if USE_SSM else 'off'}")

---
## 📚 STEP 2: LOAD HUGGINGFACE DATASET

In [ ]:
# ============================================================================
# LOAD DATASET
# ============================================================================
print(f"📚 Loading dataset: {DATASET_NAME}...")

from datasets import load_dataset

dataset_kwargs = {"split": DATASET_SPLIT}
if DATASET_SUBSET:
    dataset_kwargs["name"] = DATASET_SUBSET
if STREAMING:
    dataset_kwargs["streaming"] = True

dataset = load_dataset(DATASET_NAME, **dataset_kwargs)
print(f"  ✅ Loaded! Samples: {len(dataset):,}")

# Show sample
sample = dataset[0][TEXT_COLUMN][:200]
print(f"  📝 Sample: {sample}...")

---
## 🔤 STEP 3: CREATE TOKENIZER

In [ ]:
# ============================================================================
# CREATE TOKENIZER
# ============================================================================
print("🔤 Creating tokenizer...")

if TOKENIZER_TYPE == "hf":
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or "[PAD]"
    print(f"  HF Tokenizer: {HF_TOKENIZER_NAME}, Vocab: {tokenizer.vocab_size:,}")
else:
    # Character-level tokenizer
    chars = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,!?;:()[]{}\"' \n\t-_=/\\@#$%^&*~`|<>+"
    char_to_idx = {c: i + 1 for i, c in enumerate(chars)}
    char_to_idx["<PAD>"] = 0
    
    class SimpleTokenizer:
        def __init__(self, char_map):
            self.char_map = char_map
            self.vocab_size = len(char_map)
            self.pad_token_id = 0
        def encode(self, text):
            return [self.char_map.get(c, 0) for c in text]
        def decode(self, ids):
            rev = {v: k for k, v in self.char_map.items()}
            return "".join(rev.get(i, "") for i in ids)
    
    tokenizer = SimpleTokenizer(char_to_idx)
    print(f"  Char Tokenizer, Vocab: {tokenizer.vocab_size:,}")

VOCAB_SIZE = tokenizer.vocab_size

---
## 🔨 STEP 4: PREPARE TRAINING DATA

In [ ]:
# ============================================================================
# DATASET WRAPPER WITH FILTERS
# ============================================================================
class HFDatasetWrapper(Dataset):
    """HuggingFace Dataset wrapper with filtering and tokenization."""
    def __init__(self, hf_dataset, tokenizer, block_size=256, text_column="text",
                 min_len=None, max_len=None, max_samples=None):
        self.tokenizer = tokenizer
        self.block_size = block_size
        
        # Apply filters
        data = []
        for i, example in enumerate(hf_dataset):
            if max_samples and i >= max_samples:
                break
            text = example.get(text_column, "")
            if min_len and len(text) < min_len:
                continue
            if max_len and len(text) > max_len:
                continue
            data.append(text)
        
        print(f"  After filters: {len(data)} samples")
        
        # Tokenize and chunk
        self.tokens = []
        for text in tqdm(data, desc="Tokenizing"):
            tokens = tokenizer.encode(text)
            for i in range(0, len(tokens) - block_size, block_size // 2):
                chunk = tokens[i:i + block_size]
                if len(chunk) == block_size:
                    self.tokens.append(chunk)
        
        print(f"  Created {len(self.tokens):,} chunks")
        self.data = torch.tensor(self.tokens, dtype=torch.long) if self.tokens else torch.zeros((1, block_size), dtype=torch.long)
    
    def __len__(self):
        return len(self.tokens) if self.tokens else 1
    def __getitem__(self, idx):
        chunk = self.data[idx]
        return {"input_ids": chunk, "labels": chunk}

# Create dataset
train_data = HFDatasetWrapper(
    dataset, tokenizer,
    block_size=BLOCK_SIZE,
    text_column=TEXT_COLUMN,
    min_len=MIN_TEXT_LEN,
    max_len=MAX_TEXT_LEN,
    max_samples=MAX_SAMPLES,
)

# Train/val split
val_len = int(len(train_data) * VAL_SIZE)
train_len = len(train_data) - val_len
train_subset, val_subset = random_split(train_data, [train_len, val_len],
                                       generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"✅ Data ready! Train: {train_len:,}, Val: {val_len:,} samples")

---
## 🤖 STEP 5: CREATE MODEL

In [ ]:
# ============================================================================
# CREATE NOVALM-ULTRA MODEL
# ============================================================================
print(f"🤖 Creating model: {MODEL_NAME}")

if HAS_NOVA:
    config = UltraConfig(
        dim=DIM,
        num_layers=NUM_LAYERS,
        num_heads=NUM_HEADS,
        n_kv_heads=N_KV_HEADS,
        ffn_dim=FFN_DIM,
        vocab_size=VOCAB_SIZE,
        max_seq_len=BLOCK_SIZE,
        weight_tying=WEIGHT_TYING,
        weight_share_iterations=WEIGHT_SHARE,
        use_adaptive_router=USE_ROUTER,
        router_hidden_dim=ROUTER_DIM,
        attn_type=ATTN_TYPE,
        use_rope=USE_ROPE,
        use_rwkv=USE_RWKV,
        rwkv_frequency=RWKV_FREQ,
        use_titans_memory=USE_MEMORY,
        memory_slots=MEMORY_SLOTS,
        memory_top_k=MEMORY_TOPK,
        memory_frequency=MEMORY_FREQ,
        use_ssm_scan=USE_SSM,
        ssm_state_dim=SSM_STATE,
        ssm_frequency=SSM_FREQ,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
    )
    model = create_ultra_model(config)
    print(f"  ✅ NovaLM-ULTRA created!")
else:
    # Fallback minimal transformer
    class MinimalTransformer(nn.Module):
        def __init__(self):
            super().__init__()
            self.embed = nn.Embedding(VOCAB_SIZE, DIM)
            self.layers = nn.ModuleList([
                nn.TransformerEncoderLayer(DIM, NUM_HEADS, FFN_DIM or DIM*4, 
                    dropout=DROPOUT, activation=F.silu, batch_first=True, norm_first=True)
                for _ in range(NUM_LAYERS)
            ])
            self.norm = nn.LayerNorm(DIM)
            self.head = nn.Linear(DIM, VOCAB_SIZE, bias=False)
            self.head.weight = self.embed.weight
        def forward(self, x, state=None, return_state=False):
            mask = torch.triu(torch.ones(x.size(1), x.size(1), device=x.device) * float('-inf'), diagonal=1)
            x = self.embed(x)
            for layer in self.layers:
                x = layer(x, src_mask=mask)
            return self.head(self.norm(x)), None
        def generate(self, prompt, max_new=100, temp=0.7, top_k=50, top_p=0.9):
            self.eval()
            for _ in range(max_new):
                logits, _ = self(prompt[:, -DIM:])
                logits = logits[:, -1, :] / temp
                if top_k:
                    vals, _ = torch.topk(logits, top_k)
                    logits[logits < vals[:, -1:]] = float('-inf')
                probs = F.softmax(logits, dim=-1)
                next_tok = torch.multinomial(probs, 1)
                prompt = torch.cat([prompt, next_tok], dim=-1)
                if next_tok.item() == 0:
                    break
            return prompt
    
    model = MinimalTransformer()
    print(f"  ⚠️  Minimal transformer created (NovaLM not available)")

total_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {total_params:,}")

In [ ]:
# Device setup
if DEVICE == "auto":
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device(DEVICE)

print(f"📱 Device: {device}")
model = model.to(device)

# Multi-GPU
if MULTI_GPU and device.type == "cuda":
    if GPU_IDS:
        gpu_ids = [int(x) for x in GPU_IDS.split(",")]
    else:
        gpu_ids = list(range(torch.cuda.device_count()))
    if len(gpu_ids) > 1:
        model = nn.DataParallel(model, device_ids=gpu_ids)
        print(f"  Using GPUs: {gpu_ids}")

# Mixed precision
scaler = None
amp_dtype = None
if MIXED_PRECISION == "fp16" and device.type == "cuda":
    scaler = torch.cuda.amp.GradScaler()
    amp_dtype = torch.float16
elif MIXED_PRECISION == "bf16" and device.type == "cuda":
    amp_dtype = torch.bfloat16
elif MIXED_PRECISION == "fp16" and device.type == "mps":
    amp_dtype = torch.float16

print(f"  Mixed precision: {MIXED_PRECISION if device.type != 'cpu' else 'off (cpu)'}")

---
## 🎯 STEP 6: TRAIN THE MODEL

In [ ]:
# ============================================================================
# SETUP OPTIMIZER AND SCHEDULER
# ============================================================================
raw_model = model.module if hasattr(model, 'module') else model

# Separate weight decay
decay = []
no_decay = []
for name, p in raw_model.named_parameters():
    if p.requires_grad:
        if "norm" in name or "bias" in name:
            no_decay.append(p)
        else:
            decay.append(p)

optimizer = AdamW([
    {"params": decay, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay, "weight_decay": 0.0},
], lr=LEARNING_RATE, betas=(BETA1, BETA2))

# LR Scheduler
total_steps = len(train_loader) * EPOCHS
warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_STEPS)
cosine = CosineAnnealingLR(optimizer, T_max=total_steps - WARMUP_STEPS, eta_min=MIN_LR)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_STEPS])

print(f"🎯 Optimizer ready! Total steps: {total_steps}")
print(f"   LR: {LEARNING_RATE:.2e} -> {MIN_LR:.2e}, Warmup: {WARMUP_STEPS} steps")

In [ ]:
# ============================================================================
# TRAINING LOOP
# ============================================================================
print(f"\n{'='*60}")
print(f"🚀 TRAINING STARTED")
print(f"   Model: {MODEL_NAME}")
print(f"   Dataset: {DATASET_NAME}")
print(f"   Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, Block: {BLOCK_SIZE}")
print(f"{'='*60}\n")

# Training metrics
train_losses = []
val_losses = []
lrs = []

global_step = 0
best_loss = float("inf")

for epoch in range(EPOCHS):
    # Training
    raw_model.train()
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        
        if scaler:
            with torch.cuda.amp.autocast():
                logits, _ = model(input_ids)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1),
                                     label_smoothing=LABEL_SMOOTHING)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if GRAD_CLIP > 0:
                torch.nn.utils.clip_grad_norm_(raw_model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        elif amp_dtype:
            with torch.autocast(device_type=device.type, dtype=amp_dtype):
                logits, _ = model(input_ids)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1),
                                     label_smoothing=LABEL_SMOOTHING)
            loss.backward()
            if GRAD_CLIP > 0:
                torch.nn.utils.clip_grad_norm_(raw_model.parameters(), GRAD_CLIP)
            optimizer.step()
        else:
            logits, _ = model(input_ids)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1),
                                 label_smoothing=LABEL_SMOOTHING)
            loss.backward()
            if GRAD_CLIP > 0:
                torch.nn.utils.clip_grad_norm_(raw_model.parameters(), GRAD_CLIP)
            optimizer.step()
        
        scheduler.step()
        global_step += 1
        epoch_loss += loss.item()
        
        # Log
        current_lr = optimizer.param_groups[0]["lr"]
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{current_lr:.2e}"})
        
        if global_step % LOG_INTERVAL == 0:
            train_losses.append(loss.item())
            lrs.append(current_lr)
    
    avg_train = epoch_loss / len(train_loader)
    
    # Validation
    raw_model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            labs = batch["labels"].to(device)
            logs, _ = model(ids)
            val_loss += F.cross_entropy(logs.view(-1, logs.size(-1)), labs.view(-1)).item()
    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)
    
    print(f"\n📊 Epoch {epoch+1}: Train Loss = {avg_train:.4f}, Val Loss = {avg_val:.4f}")
    
    # Save best
    if avg_val < best_loss:
        best_loss = avg_val
        save_path = Path(SAVE_DIR) / MODEL_NAME
        save_path.mkdir(parents=True, exist_ok=True)
        torch.save({
            "model_state": raw_model.state_dict(),
            "config": {"dim": DIM, "layers": NUM_LAYERS, "heads": NUM_HEADS, "vocab": VOCAB_SIZE},
        }, save_path / "best_model.pt")
        print(f"  🏆 New best model saved!")

print(f"\n{'='*60}")
print(f"✅ TRAINING COMPLETE!")
print(f"   Best validation loss: {best_loss:.4f}")
print(f"{'='*60}")

---
## 📊 STEP 7: VISUALIZE TRAINING

In [ ]:
# ============================================================================
# PLOT TRAINING CURVES
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curve
axes[0].plot(train_losses, label="Train Loss", alpha=0.7)
if val_losses:
    # Plot validation at epoch boundaries
    val_x = [len(train_losses) * (i+1) // EPOCHS for i in range(len(val_losses))]
    axes[0].plot(val_x, val_losses, "ro-", label="Val Loss", markersize=8)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Learning rate
axes[1].plot(lrs, color="green", alpha=0.7)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("Learning Rate Schedule")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📊 Final train loss: {train_losses[-1]:.4f}")
print(f"📊 Best val loss: {best_loss:.4f}")

---
## ✍️ STEP 8: GENERATE TEXT

In [ ]:
# ============================================================================
# GENERATE TEXT WITH TRAINED MODEL
# ============================================================================
if GENERATE_AFTER_TRAIN:
    print(f"✍️ Generating text...")
    print(f"   Prompt: '{PROMPT}'")
    
    # Encode prompt
    prompt_ids = torch.tensor([tokenizer.encode(PROMPT)], dtype=torch.long).to(device)
    
    # Generate
    raw_model.eval()
    with torch.no_grad():
        output_ids = raw_model.generate(
            prompt_ids,
            max_new_tokens=GEN_TOKENS,
            temperature=GEN_TEMP,
            top_k=GEN_TOP_K,
            top_p=GEN_TOP_P,
        )
    
    # Decode
    output_text = tokenizer.decode(output_ids[0].tolist())
    
    print(f"\n{'─'*60}")
    print(f"   {output_text}")
    print(f"{'─'*60}")
else:
    print("Skipping generation (GENERATE_AFTER_TRAIN = False)")

---
## 💾 STEP 9: SAVE MODEL

In [ ]:
# ============================================================================
# SAVE FINAL MODEL
# ============================================================================
save_path = Path(SAVE_DIR) / MODEL_NAME
save_path.mkdir(parents=True, exist_ok=True)

final_path = save_path / "final_model.pt"
torch.save({
    "model_state": raw_model.state_dict(),
    "config": {
        "dim": DIM,
        "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS,
        "vocab_size": VOCAB_SIZE,
        "model_name": MODEL_NAME,
    },
}, final_path)

print(f"💾 Model saved to: {final_path}")
print(f"\n📁 All files in {save_path}:")
for f in save_path.glob("*"):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   {f.name} ({size_mb:.1f} MB)")

---
## 🔄 STEP 10: LOAD MODEL & CONTINUE (Bonus)

In [ ]:
# ============================================================================
# LOAD SAVED MODEL
# ============================================================================
def load_model(checkpoint_path):
    """Load a saved model from checkpoint."""
    print(f"📂 Loading model from: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    
    # Recreate model with saved config
    cfg = checkpoint["config"]
    if HAS_NOVA:
        ultra_cfg = UltraConfig(
            dim=cfg.get("dim", DIM),
            num_layers=cfg.get("num_layers", NUM_LAYERS),
            num_heads=cfg.get("num_heads", NUM_HEADS),
            vocab_size=cfg.get("vocab_size", VOCAB_SIZE),
        )
        loaded = create_ultra_model(ultra_cfg)
    else:
        loaded = MinimalTransformer()
    
    loaded.load_state_dict(checkpoint["model_state"])
    loaded = loaded.to(device)
    loaded.eval()
    
    print(f"  ✅ Model loaded! Params: {sum(p.numel() for p in loaded.parameters()):,}")
    return loaded

# Example usage:
# loaded_model = load_model(str(final_path))
# # Generate with loaded model
# prompt = "Once upon a time"
# prompt_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long).to(device)
# output = loaded_model.generate(prompt_ids, max_new_tokens=50)
# print(tokenizer.decode(output[0].tolist()))

---
## 🎉 DONE! NEXT STEPS

### Try Different Configurations:

```python
# Fast testing
DIM = 256; NUM_LAYERS = 4; EPOCHS = 1; DATASET_NAME = "roneneldan/TinyStories"

# Small LLM (good for CPU)
DIM = 512; NUM_LAYERS = 8; EPOCHS = 10; DATASET_NAME = "roneneldan/TinyStories"

# Medium LLM (needs GPU)
DIM = 768; NUM_LAYERS = 12; DATASET_NAME = "HuggingFaceFW/fineweb"
BATCH_SIZE = 16; USE_HF_TOKENIZER = True; HF_TOKENIZER_NAME = "gpt2"

# Large LLM (multi-GPU)
DIM = 2048; NUM_LAYERS = 24; MULTI_GPU = True
DATASET_NAME = "HuggingFaceFW/fineweb-edu"
BATCH_SIZE = 8; BLOCK_SIZE = 2048; MIXED_PRECISION = "bf16"
```

### CLI Version:
```bash
# Same training via command line
python scripts/train_ultra_complete.py --dataset roneneldan/TinyStories --dim 512 --layers 8 --epochs 3 --model-name "MyNovaLM"
```